In [2]:
# # STEP 02.7 — FINAL CROSS-MODAL PAIR DATASET CREATION

# ## Objective

# Create the final scientifically validated cross-modal pair dataset by
# combining positive and negative cross-modal pairs constructed in previous steps.

# The final dataset will serve as the input for subsequent cross-modal
# representation learning.

# ### Methodological Principles

# - Use only previously constructed and validated pairs
# - Preserve knowledge-level methodology
# - Do not assume patient correspondence
# - Maintain positive/negative class balance
# - Preserve modality balance
# - Shuffle reproducibly
# - Create controlled train/validation partitions
# - Preserve pair metadata for traceability

# ### Expected Input

# - Positive pairs from Step 02.4
# - Negative pairs from Step 02.5
# - Validation evidence from Step 02.6

# ### Expected Output

# A unified final cross-modal pair dataset containing:

# - Source embeddings
# - Target embeddings
# - Pair labels
# - Pair metadata
# - Train/validation split information

# ### Important Note

# A pair represents a semantic relationship between independently learned
# modality representations.

# It does NOT represent measurements from the same patient.

In [3]:
# ============================================================
# CELL 2 — IMPORTS AND INITIALIZATION
# ============================================================

import os
import random
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# REPRODUCIBILITY
# ------------------------------------------------------------

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# ------------------------------------------------------------
# DISPLAY SETTINGS
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

print("=" * 100)
print("STEP 02.7 — FINAL CROSS-MODAL PAIR DATASET CREATION")
print("=" * 100)

print(f"\nRandom Seed: {RANDOM_SEED}")
print("\nInitialization completed successfully.")

STEP 02.7 — FINAL CROSS-MODAL PAIR DATASET CREATION

Random Seed: 42

Initialization completed successfully.


In [4]:
# ============================================================
# CELL 3 — DEFINE PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parents[2]

PAIR_DIR = (
    PROJECT_ROOT /
    "06_cross_modal_learning" /
    "02_cross_modal_pairing" /
    "pairs"
)

REPORT_DIR = (
    PROJECT_ROOT /
    "06_cross_modal_learning" /
    "02_cross_modal_pairing" /
    "reports"
)

# ------------------------------------------------------------
# INPUT FILES
# ------------------------------------------------------------

POSITIVE_PAIR_PATH = (
    PAIR_DIR /
    "02_04_positive_cross_modal_embedding_pairs.npz"
)

POSITIVE_METADATA_PATH = (
    PAIR_DIR /
    "02_04_positive_cross_modal_pair_metadata.csv"
)

NEGATIVE_PAIR_PATH = (
    PAIR_DIR /
    "02_05_negative_cross_modal_embedding_pairs.npz"
)

NEGATIVE_METADATA_PATH = (
    PAIR_DIR /
    "02_05_negative_cross_modal_pair_metadata.csv"
)

# ------------------------------------------------------------
# OUTPUT DIRECTORY
# ------------------------------------------------------------

FINAL_PAIR_DIR = PAIR_DIR / "final_dataset"
FINAL_PAIR_DIR.mkdir(parents=True, exist_ok=True)

REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT ROOT:")
print(PROJECT_ROOT)

print("\nPositive pair file:")
print(POSITIVE_PAIR_PATH)

print("\nNegative pair file:")
print(NEGATIVE_PAIR_PATH)

print("\nFinal output directory:")
print(FINAL_PAIR_DIR)

PROJECT ROOT:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening

Positive pair file:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\02_04_positive_cross_modal_embedding_pairs.npz

Negative pair file:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\02_05_negative_cross_modal_embedding_pairs.npz

Final output directory:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\final_dataset


In [5]:
# ============================================================
# CELL 4 — VERIFY INPUT ARTIFACT AVAILABILITY
# ============================================================

required_files = {
    "Positive Embedding Pairs": POSITIVE_PAIR_PATH,
    "Positive Pair Metadata": POSITIVE_METADATA_PATH,
    "Negative Embedding Pairs": NEGATIVE_PAIR_PATH,
    "Negative Pair Metadata": NEGATIVE_METADATA_PATH
}

print("=" * 100)
print("INPUT ARTIFACT VERIFICATION")
print("=" * 100)

all_files_available = True

for name, path in required_files.items():

    exists = path.exists()

    print(
        f"{name:<30}: "
        f"{'FOUND ✓' if exists else 'MISSING ✗'}"
    )

    if not exists:
        all_files_available = False

if not all_files_available:
    raise FileNotFoundError(
        "\nOne or more required pairing artifacts are missing. "
        "Complete Steps 02.4 and 02.5 before proceeding."
    )

print("\n✓ All required input artifacts are available.")

INPUT ARTIFACT VERIFICATION
Positive Embedding Pairs      : FOUND ✓
Positive Pair Metadata        : FOUND ✓
Negative Embedding Pairs      : FOUND ✓
Negative Pair Metadata        : FOUND ✓

✓ All required input artifacts are available.


In [6]:
# ============================================================
# CELL 5 — LOAD POSITIVE AND NEGATIVE PAIR DATASETS
# ============================================================

print("=" * 100)
print("LOADING PAIR DATASETS")
print("=" * 100)

# ------------------------------------------------------------
# LOAD POSITIVE PAIRS
# ------------------------------------------------------------

positive_data = np.load(POSITIVE_PAIR_PATH)

positive_source = positive_data["source_embeddings"]
positive_target = positive_data["target_embeddings"]

positive_metadata = pd.read_csv(POSITIVE_METADATA_PATH)


# ------------------------------------------------------------
# LOAD NEGATIVE PAIRS
# ------------------------------------------------------------

negative_data = np.load(NEGATIVE_PAIR_PATH)

negative_source = negative_data["source_embeddings"]
negative_target = negative_data["target_embeddings"]

negative_metadata = pd.read_csv(NEGATIVE_METADATA_PATH)


# ------------------------------------------------------------
# DISPLAY STRUCTURE
# ------------------------------------------------------------

print("\nPOSITIVE PAIRS")
print("-" * 60)
print(f"Source shape : {positive_source.shape}")
print(f"Target shape : {positive_target.shape}")
print(f"Metadata rows: {len(positive_metadata)}")

print("\nNEGATIVE PAIRS")
print("-" * 60)
print(f"Source shape : {negative_source.shape}")
print(f"Target shape : {negative_target.shape}")
print(f"Metadata rows: {len(negative_metadata)}")

LOADING PAIR DATASETS

POSITIVE PAIRS
------------------------------------------------------------
Source shape : (174, 32)
Target shape : (174, 32)
Metadata rows: 174

NEGATIVE PAIRS
------------------------------------------------------------
Source shape : (174, 32)
Target shape : (174, 32)
Metadata rows: 174


In [7]:
# ============================================================
# CELL 6 — VALIDATE PAIR-METADATA ALIGNMENT
# ============================================================

print("=" * 100)
print("PAIR-METADATA ALIGNMENT VALIDATION")
print("=" * 100)

validation_checks = {

    "Positive Source vs Target":
        positive_source.shape[0] == positive_target.shape[0],

    "Positive Source vs Metadata":
        positive_source.shape[0] == len(positive_metadata),

    "Negative Source vs Target":
        negative_source.shape[0] == negative_target.shape[0],

    "Negative Source vs Metadata":
        negative_source.shape[0] == len(negative_metadata),

    "Positive Embedding Dimension":
        positive_source.shape[1] == 32 and positive_target.shape[1] == 32,

    "Negative Embedding Dimension":
        negative_source.shape[1] == 32 and negative_target.shape[1] == 32
}

all_passed = True

for check_name, result in validation_checks.items():

    status = "PASS ✓" if result else "FAIL ✗"

    print(f"{check_name:<40}: {status}")

    if not result:
        all_passed = False

if not all_passed:
    raise ValueError(
        "\nDataset alignment validation failed."
    )

print("\n✓ All pair datasets are structurally aligned.")

PAIR-METADATA ALIGNMENT VALIDATION
Positive Source vs Target               : PASS ✓
Positive Source vs Metadata             : PASS ✓
Negative Source vs Target               : PASS ✓
Negative Source vs Metadata             : PASS ✓
Positive Embedding Dimension            : PASS ✓
Negative Embedding Dimension            : PASS ✓

✓ All pair datasets are structurally aligned.


In [8]:
# ============================================================
# CELL 7 — CREATE UNIFIED PAIR LABELS
# ============================================================

print("=" * 100)
print("CREATING UNIFIED PAIR LABELS")
print("=" * 100)

# ------------------------------------------------------------
# POSITIVE = 1
# NEGATIVE = 0
# ------------------------------------------------------------

positive_labels = np.ones(
    len(positive_source),
    dtype=np.int64
)

negative_labels = np.zeros(
    len(negative_source),
    dtype=np.int64
)

print(
    f"Positive labels created: "
    f"{len(positive_labels)} samples → Label 1"
)

print(
    f"Negative labels created: "
    f"{len(negative_labels)} samples → Label 0"
)

print("\nLabel semantics:")
print("1 → Same-Class / Positive Relationship")
print("0 → Different-Class / Negative Relationship")

CREATING UNIFIED PAIR LABELS
Positive labels created: 174 samples → Label 1
Negative labels created: 174 samples → Label 0

Label semantics:
1 → Same-Class / Positive Relationship
0 → Different-Class / Negative Relationship


In [9]:
# ============================================================
# CELL 8 — COMBINE POSITIVE AND NEGATIVE PAIRS
# ============================================================

print("=" * 100)
print("COMBINING CROSS-MODAL PAIRS")
print("=" * 100)

# ------------------------------------------------------------
# COMBINE SOURCE EMBEDDINGS
# ------------------------------------------------------------

final_source_embeddings = np.vstack([
    positive_source,
    negative_source
])

# ------------------------------------------------------------
# COMBINE TARGET EMBEDDINGS
# ------------------------------------------------------------

final_target_embeddings = np.vstack([
    positive_target,
    negative_target
])

# ------------------------------------------------------------
# COMBINE PAIR LABELS
# ------------------------------------------------------------

final_pair_labels = np.concatenate([
    positive_labels,
    negative_labels
])

print("\nFINAL EMBEDDING DATASET STRUCTURE")
print("-" * 60)

print(
    f"Source embeddings : "
    f"{final_source_embeddings.shape}"
)

print(
    f"Target embeddings : "
    f"{final_target_embeddings.shape}"
)

print(
    f"Pair labels       : "
    f"{final_pair_labels.shape}"
)

print(
    f"\nTotal pairs       : "
    f"{len(final_pair_labels)}"
)

COMBINING CROSS-MODAL PAIRS

FINAL EMBEDDING DATASET STRUCTURE
------------------------------------------------------------
Source embeddings : (348, 32)
Target embeddings : (348, 32)
Pair labels       : (348,)

Total pairs       : 348


In [10]:
# ============================================================
# CELL 9 — COMBINE AND STANDARDIZE PAIR METADATA
# ============================================================

print("=" * 100)
print("COMBINING PAIR METADATA")
print("=" * 100)

# ------------------------------------------------------------
# CREATE CLEAN COPIES
# ------------------------------------------------------------

positive_metadata_final = positive_metadata.copy()
negative_metadata_final = negative_metadata.copy()


# ------------------------------------------------------------
# STANDARDIZE PAIR TYPE
# ------------------------------------------------------------

positive_metadata_final["final_pair_type"] = "positive"
negative_metadata_final["final_pair_type"] = "negative"


# ------------------------------------------------------------
# STANDARDIZE NUMERICAL PAIR LABEL
# ------------------------------------------------------------

positive_metadata_final["final_pair_label"] = 1
negative_metadata_final["final_pair_label"] = 0


# ------------------------------------------------------------
# COMBINE METADATA
# ------------------------------------------------------------

final_metadata = pd.concat(
    [
        positive_metadata_final,
        negative_metadata_final
    ],
    axis=0,
    ignore_index=True,
    sort=False
)


print(f"Positive metadata rows: {len(positive_metadata_final)}")
print(f"Negative metadata rows: {len(negative_metadata_final)}")

print("\nFinal metadata shape:")
print(final_metadata.shape)

print("\nAvailable columns:")
print(list(final_metadata.columns))

print("\nFirst 5 rows:")
display(final_metadata.head())

COMBINING PAIR METADATA
Positive metadata rows: 174
Negative metadata rows: 174

Final metadata shape:
(348, 14)

Available columns:
['source_modality', 'target_modality', 'source_index', 'target_index', 'source_class', 'target_class', 'class_name', 'pair_label', 'pair_type', 'relationship', 'final_pair_type', 'final_pair_label', 'semantic_relationship', 'is_different_class']

First 5 rows:


,source_modality,target_modality,source_index,target_index,source_class,target_class,class_name,pair_label,pair_type,relationship,final_pair_type,final_pair_label,semantic_relationship,is_different_class
0,Genetic,Behavioral,19,158,0,0,NON-ASD,1.0,Positive,Same-Class Cross-Modal,positive,1,NaN,NaN
1,Genetic,Behavioral,28,822,0,0,NON-ASD,1.0,Positive,Same-Class Cross-Modal,positive,1,NaN,NaN
2,Genetic,Behavioral,3,194,0,0,NON-ASD,1.0,Positive,Same-Class Cross-Modal,positive,1,NaN,NaN
3,Genetic,Behavioral,14,131,0,0,NON-ASD,1.0,Positive,Same-Class Cross-Modal,positive,1,NaN,NaN
4,Genetic,Behavioral,7,563,0,0,NON-ASD,1.0,Positive,Same-Class Cross-Modal,positive,1,NaN,NaN


In [11]:
# ============================================================
# CELL 10 — FINAL DATASET BALANCE VERIFICATION
# ============================================================

print("=" * 100)
print("FINAL PAIR DATASET BALANCE")
print("=" * 100)

# ------------------------------------------------------------
# POSITIVE / NEGATIVE BALANCE
# ------------------------------------------------------------

label_counts = pd.Series(
    final_pair_labels
).value_counts().sort_index()

balance_summary = pd.DataFrame({
    "Pair_Label": label_counts.index,
    "Pair_Type": [
        "Negative (Different-Class)",
        "Positive (Same-Class)"
    ],
    "Count": label_counts.values,
    "Percentage": (
        label_counts.values /
        len(final_pair_labels) * 100
    )
})

display(balance_summary)


# ------------------------------------------------------------
# BALANCE RATIO
# ------------------------------------------------------------

balance_ratio = (
    label_counts.max() /
    label_counts.min()
)

print(
    f"\nPair Balance Ratio: "
    f"{balance_ratio:.4f}"
)

if balance_ratio == 1:
    print("✓ Perfect Positive-Negative Balance")
else:
    print("⚠ Dataset is not perfectly balanced")

FINAL PAIR DATASET BALANCE


,Pair_Label,Pair_Type,Count,Percentage
0,0,Negative (Different-Class),174,50.0
1,1,Positive (Same-Class),174,50.0



Pair Balance Ratio: 1.0000
✓ Perfect Positive-Negative Balance


In [12]:
# ============================================================
# CELL 11 — REPRODUCIBLE DATASET SHUFFLING
# ============================================================

print("=" * 100)
print("REPRODUCIBLE DATASET SHUFFLING")
print("=" * 100)

# ------------------------------------------------------------
# CREATE RANDOM PERMUTATION
# ------------------------------------------------------------

rng = np.random.default_rng(RANDOM_SEED)

shuffle_indices = rng.permutation(
    len(final_pair_labels)
)


# ------------------------------------------------------------
# APPLY SAME SHUFFLE TO ALL DATA STRUCTURES
# ------------------------------------------------------------

final_source_embeddings = (
    final_source_embeddings[shuffle_indices]
)

final_target_embeddings = (
    final_target_embeddings[shuffle_indices]
)

final_pair_labels = (
    final_pair_labels[shuffle_indices]
)

final_metadata = (
    final_metadata
    .iloc[shuffle_indices]
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# VALIDATE ALIGNMENT
# ------------------------------------------------------------

assert (
    len(final_source_embeddings)
    ==
    len(final_target_embeddings)
    ==
    len(final_pair_labels)
    ==
    len(final_metadata)
)

print(
    f"✓ {len(final_pair_labels)} pairs shuffled successfully"
)

print(
    f"✓ Random seed {RANDOM_SEED} ensures reproducibility"
)

REPRODUCIBLE DATASET SHUFFLING
✓ 348 pairs shuffled successfully
✓ Random seed 42 ensures reproducibility


In [13]:
# ============================================================
# CELL 12 — TRAIN / VALIDATION SPLIT
# ============================================================

print("=" * 100)
print("CREATING TRAIN / VALIDATION PARTITIONS")
print("=" * 100)

VALIDATION_SIZE = 0.20

all_indices = np.arange(
    len(final_pair_labels)
)

train_indices, validation_indices = train_test_split(
    all_indices,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_SEED,
    stratify=final_pair_labels
)


# ------------------------------------------------------------
# CREATE TRAIN DATASET
# ------------------------------------------------------------

train_source = (
    final_source_embeddings[train_indices]
)

train_target = (
    final_target_embeddings[train_indices]
)

train_labels = (
    final_pair_labels[train_indices]
)

train_metadata = (
    final_metadata
    .iloc[train_indices]
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# CREATE VALIDATION DATASET
# ------------------------------------------------------------

validation_source = (
    final_source_embeddings[validation_indices]
)

validation_target = (
    final_target_embeddings[validation_indices]
)

validation_labels = (
    final_pair_labels[validation_indices]
)

validation_metadata = (
    final_metadata
    .iloc[validation_indices]
    .reset_index(drop=True)
)


print("\nDATASET SPLIT SUMMARY")
print("-" * 60)

print(f"Total pairs      : {len(final_pair_labels)}")
print(f"Training pairs   : {len(train_labels)}")
print(f"Validation pairs : {len(validation_labels)}")

print(
    f"\nTraining percentage   : "
    f"{len(train_labels)/len(final_pair_labels)*100:.2f}%"
)

print(
    f"Validation percentage : "
    f"{len(validation_labels)/len(final_pair_labels)*100:.2f}%"
)

CREATING TRAIN / VALIDATION PARTITIONS

DATASET SPLIT SUMMARY
------------------------------------------------------------
Total pairs      : 348
Training pairs   : 278
Validation pairs : 70

Training percentage   : 79.89%
Validation percentage : 20.11%


In [14]:
# ============================================================
# CELL 13 — TRAIN / VALIDATION BALANCE VALIDATION
# ============================================================

print("=" * 100)
print("TRAIN / VALIDATION BALANCE VALIDATION")
print("=" * 100)

train_distribution = pd.Series(
    train_labels
).value_counts().sort_index()

validation_distribution = pd.Series(
    validation_labels
).value_counts().sort_index()


split_summary = pd.DataFrame({

    "Pair_Type": [
        "Negative",
        "Positive"
    ],

    "Train_Count": [
        train_distribution.get(0, 0),
        train_distribution.get(1, 0)
    ],

    "Validation_Count": [
        validation_distribution.get(0, 0),
        validation_distribution.get(1, 0)
    ]

})

split_summary["Train_Percentage"] = (
    split_summary["Train_Count"] /
    len(train_labels) * 100
)

split_summary["Validation_Percentage"] = (
    split_summary["Validation_Count"] /
    len(validation_labels) * 100
)

display(split_summary.round(2))

TRAIN / VALIDATION BALANCE VALIDATION


,Pair_Type,Train_Count,Validation_Count,Train_Percentage,Validation_Percentage
0,Negative,139,35,50.0,50.0
1,Positive,139,35,50.0,50.0


In [16]:
# ============================================================
# CELL 14 — MODALITY REPRESENTATION VALIDATION
# ============================================================

print("=" * 100)
print("MODALITY REPRESENTATION ACROSS DATA SPLITS")
print("=" * 100)

# ------------------------------------------------------------
# TRAIN MODALITY COMBINATIONS
# ------------------------------------------------------------

train_modality_counts = (
    train_metadata
    .groupby(
        ["source_modality", "target_modality"]
    )
    .size()
    .reset_index(name="Train_Count")
)


# ------------------------------------------------------------
# VALIDATION MODALITY COMBINATIONS
# ------------------------------------------------------------

validation_modality_counts = (
    validation_metadata
    .groupby(
        ["source_modality", "target_modality"]
    )
    .size()
    .reset_index(name="Validation_Count")
)


# ------------------------------------------------------------
# MERGE
# ------------------------------------------------------------

modality_split_summary = pd.merge(
    train_modality_counts,
    validation_modality_counts,
    on=[
        "source_modality",
        "target_modality"
    ],
    how="outer"
).fillna(0)


modality_split_summary["Total"] = (
    modality_split_summary["Train_Count"]
    +
    modality_split_summary["Validation_Count"]
)

display(modality_split_summary)

MODALITY REPRESENTATION ACROSS DATA SPLITS


,source_modality,target_modality,Train_Count,Validation_Count,Total
0,Behavioral,Environmental,96,20,116
1,Genetic,Behavioral,94,22,116
2,Genetic,Environmental,88,28,116


In [17]:
# ============================================================
# CELL 15 — TRAIN / VALIDATION PAIR OVERLAP DIAGNOSTIC
# ============================================================

print("=" * 100)
print("PAIR OVERLAP DIAGNOSTIC")
print("=" * 100)

# ------------------------------------------------------------
# CREATE PAIR IDENTIFIERS
# ------------------------------------------------------------

def create_pair_ids(metadata):

    return set(
        metadata.apply(
            lambda row:
            f"{row['source_modality']}|"
            f"{row['source_index']}|"
            f"{row['target_modality']}|"
            f"{row['target_index']}",
            axis=1
        )
    )


train_pair_ids = create_pair_ids(train_metadata)
validation_pair_ids = create_pair_ids(validation_metadata)

overlap = (
    train_pair_ids
    .intersection(validation_pair_ids)
)

print(
    f"Training unique pair IDs   : "
    f"{len(train_pair_ids)}"
)

print(
    f"Validation unique pair IDs : "
    f"{len(validation_pair_ids)}"
)

print(
    f"Exact pair overlap         : "
    f"{len(overlap)}"
)

if len(overlap) == 0:
    print("\n✓ No exact pair duplication across splits.")
else:
    print(
        "\n⚠ Exact pair overlap detected."
    )

PAIR OVERLAP DIAGNOSTIC
Training unique pair IDs   : 278
Validation unique pair IDs : 70
Exact pair overlap         : 0

✓ No exact pair duplication across splits.


In [18]:
# ============================================================
# CELL 16 — FINAL DATASET SUMMARY
# ============================================================

print("=" * 100)
print("FINAL CROSS-MODAL PAIR DATASET SUMMARY")
print("=" * 100)

final_summary = pd.DataFrame([

    {
        "Metric": "Total Cross-Modal Pairs",
        "Value": len(final_pair_labels)
    },

    {
        "Metric": "Positive Same-Class Pairs",
        "Value": int(np.sum(final_pair_labels == 1))
    },

    {
        "Metric": "Negative Different-Class Pairs",
        "Value": int(np.sum(final_pair_labels == 0))
    },

    {
        "Metric": "Embedding Dimension",
        "Value": final_source_embeddings.shape[1]
    },

    {
        "Metric": "Source Embedding Shape",
        "Value": str(final_source_embeddings.shape)
    },

    {
        "Metric": "Target Embedding Shape",
        "Value": str(final_target_embeddings.shape)
    },

    {
        "Metric": "Training Pairs",
        "Value": len(train_labels)
    },

    {
        "Metric": "Validation Pairs",
        "Value": len(validation_labels)
    },

    {
        "Metric": "Positive-Negative Balance Ratio",
        "Value": balance_ratio
    },

    {
        "Metric": "Random Seed",
        "Value": RANDOM_SEED
    },

    {
        "Metric": "Patient-Level Matching",
        "Value": "Not Used"
    },

    {
        "Metric": "Methodology",
        "Value": "Knowledge-Level Cross-Modal Pairing"
    }

])

display(final_summary)

FINAL CROSS-MODAL PAIR DATASET SUMMARY


,Metric,Value
0,Total Cross-Modal Pairs,348
1,Positive Same-Class Pairs,174
2,Negative Different-Class Pairs,174
3,Embedding Dimension,32
4,Source Embedding Shape,"(348, 32)"
5,Target Embedding Shape,"(348, 32)"
6,Training Pairs,278
7,Validation Pairs,70
8,Positive-Negative Balance Ratio,1.0
9,Random Seed,42


In [19]:
# ============================================================
# CELL 17 — SAVE FINAL PAIR DATASET ARTIFACTS
# ============================================================

print("=" * 100)
print("SAVING FINAL CROSS-MODAL PAIR DATASET")
print("=" * 100)

# ------------------------------------------------------------
# SAVE COMPLETE DATASET
# ------------------------------------------------------------

np.savez_compressed(

    FINAL_PAIR_DIR /
    "02_07_final_cross_modal_pair_dataset.npz",

    source_embeddings=final_source_embeddings,
    target_embeddings=final_target_embeddings,
    pair_labels=final_pair_labels

)


# ------------------------------------------------------------
# SAVE TRAIN DATASET
# ------------------------------------------------------------

np.savez_compressed(

    FINAL_PAIR_DIR /
    "02_07_train_cross_modal_pairs.npz",

    source_embeddings=train_source,
    target_embeddings=train_target,
    pair_labels=train_labels

)


# ------------------------------------------------------------
# SAVE VALIDATION DATASET
# ------------------------------------------------------------

np.savez_compressed(

    FINAL_PAIR_DIR /
    "02_07_validation_cross_modal_pairs.npz",

    source_embeddings=validation_source,
    target_embeddings=validation_target,
    pair_labels=validation_labels

)


# ------------------------------------------------------------
# SAVE METADATA
# ------------------------------------------------------------

final_metadata.to_csv(

    FINAL_PAIR_DIR /
    "02_07_final_pair_metadata.csv",

    index=False
)

train_metadata.to_csv(

    FINAL_PAIR_DIR /
    "02_07_train_pair_metadata.csv",

    index=False
)

validation_metadata.to_csv(

    FINAL_PAIR_DIR /
    "02_07_validation_pair_metadata.csv",

    index=False
)


# ------------------------------------------------------------
# SAVE REPORTS
# ------------------------------------------------------------

final_summary.to_csv(

    REPORT_DIR /
    "02_07_final_pair_dataset_summary.csv",

    index=False
)

split_summary.to_csv(

    REPORT_DIR /
    "02_07_train_validation_distribution.csv",

    index=False
)

modality_split_summary.to_csv(

    REPORT_DIR /
    "02_07_modality_split_distribution.csv",

    index=False
)


print("\n✓ Complete pair dataset saved")
print("✓ Training dataset saved")
print("✓ Validation dataset saved")
print("✓ Final metadata saved")
print("✓ Train metadata saved")
print("✓ Validation metadata saved")
print("✓ Dataset summary saved")
print("✓ Split distribution report saved")

SAVING FINAL CROSS-MODAL PAIR DATASET

✓ Complete pair dataset saved
✓ Training dataset saved
✓ Validation dataset saved
✓ Final metadata saved
✓ Train metadata saved
✓ Validation metadata saved
✓ Dataset summary saved
✓ Split distribution report saved


In [20]:
# ============================================================
# CELL 18 — FINAL SCIENTIFIC CONCLUSION
# ============================================================

print("\n")
print("=" * 100)
print("STEP 02.7 — FINAL CROSS-MODAL PAIR DATASET CREATION")
print("=" * 100)

print("""
FINAL DATASET STATUS
----------------------------------------------------------------------------------------------------

Total Cross-Modal Pairs:
""", len(final_pair_labels))

print("""
Positive Same-Class Pairs:
""", np.sum(final_pair_labels == 1))

print("""
Negative Different-Class Pairs:
""", np.sum(final_pair_labels == 0))

print("""
Embedding Dimension:
""", final_source_embeddings.shape[1])

print("""
Train / Validation Split:
""")

print(f"Training pairs   : {len(train_labels)}")
print(f"Validation pairs : {len(validation_labels)}")

print("""
====================================================================================================
SCIENTIFIC INTERPRETATION
====================================================================================================

The final cross-modal pairing dataset combines scientifically validated
same-class and different-class relationships between independently
learned modality representations.

Positive pairs represent semantic compatibility between representations
belonging to the same diagnostic class.

Negative pairs represent controlled semantic contrast between
representations belonging to different diagnostic classes.

The dataset preserves:

✓ Positive-negative balance
✓ Modality representation balance
✓ Class-conditional semantic relationships
✓ Reproducible sampling
✓ Pair-level metadata traceability
✓ Knowledge-level methodology

The dataset does NOT assume:

✗ Shared patient identities
✗ Patient-level correspondence
✗ Biological equivalence between paired samples

Each pair represents a knowledge-level semantic training relationship
between independently learned representation spaces.

====================================================================================================
STEP 02.7 COMPLETED
====================================================================================================

✓ FINAL PAIR DATASET CREATED
✓ POSITIVE AND NEGATIVE PAIRS COMBINED
✓ REPRODUCIBLE SHUFFLING APPLIED
✓ TRAIN / VALIDATION PARTITIONS CREATED
✓ PAIR METADATA PRESERVED
✓ KNOWLEDGE-LEVEL METHODOLOGY PRESERVED

NEXT PHASE:
PHASE 03 — CROSS-MODAL REPRESENTATION LEARNING
====================================================================================================
""")



STEP 02.7 — FINAL CROSS-MODAL PAIR DATASET CREATION

FINAL DATASET STATUS
----------------------------------------------------------------------------------------------------

Total Cross-Modal Pairs:
 348

Positive Same-Class Pairs:
 174

Negative Different-Class Pairs:
 174

Embedding Dimension:
 32

Train / Validation Split:

Training pairs   : 278
Validation pairs : 70

SCIENTIFIC INTERPRETATION

The final cross-modal pairing dataset combines scientifically validated
same-class and different-class relationships between independently
learned modality representations.

Positive pairs represent semantic compatibility between representations
belonging to the same diagnostic class.

Negative pairs represent controlled semantic contrast between
representations belonging to different diagnostic classes.

The dataset preserves:

✓ Positive-negative balance
✓ Modality representation balance
✓ Class-conditional semantic relationships
✓ Reproducible sampling
✓ Pair-level metadata traceabili

In [1]:
# ============================================================
# CHECK ALL PHASE 02 FINAL DATASET ARTIFACTS
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening"
)

PHASE02_DIR = (
    PROJECT_ROOT
    / "06_cross_modal_learning"
    / "02_cross_modal_pairing"
)

print("=" * 100)
print("SEARCHING FOR PHASE 02.7 FINAL DATASET ARTIFACTS")
print("=" * 100)

for file in sorted(PHASE02_DIR.rglob("*")):
    if file.is_file() and "02_07" in file.name:
        print(file)

SEARCHING FOR PHASE 02.7 FINAL DATASET ARTIFACTS
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\notebooks\.ipynb_checkpoints\02_07_final_pair_dataset_creation-checkpoint.ipynb
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\notebooks\02_07_final_pair_dataset_creation.ipynb
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\final_dataset\02_07_final_cross_modal_pair_dataset.npz
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\final_dataset\02_07_final_pair_metadata.csv
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\final_dataset\02_07_train_cross_modal_pairs.npz
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\0